# Keras/TensorFlow — Chapter 11: Project — Regression of Boston House Prices


## 1. Bài toán và dữ liệu

- Bộ dữ liệu **Boston House Price**: 13 biến số học (tội phạm, số phòng, khoảng cách khu công nghiệp...), dự đoán giá nhà trung vị (nghìn USD). 506 mẫu. Benchmark MSE tham chiếu: **~20** (tương đương sai số ~$4,500 nếu lấy căn bậc hai).
- Nạp bằng `pandas.read_csv(..., delim_whitespace=True)`.

## 2. Model baseline và đánh giá bằng KerasRegressor

- Kiến trúc tối giản: 13 → 13 (ReLU) → 1. Loss: `mean_squared_error`.
- `KerasRegressor` (SciKeras) cho bài toán hồi quy.
- `cross_val_score(..., scoring='neg_mean_squared_error')` — scikit-learn luôn tối đa hoá điểm số, nên MSE được đảo dấu; **bỏ qua dấu âm** khi đọc kết quả.
- **Kết quả thật**: **-32.65 (23.33) MSE** → đọc là MSE trung bình 32.65, độ lệch chuẩn 23.33.

## 3. Chuẩn hoá dữ liệu bằng Pipeline

- `Pipeline([('standardize', StandardScaler()), ('mlp', KerasRegressor(...))])`.
- **Kết quả**: **-29.54 (27.87) MSE** — mean cải thiện (32.65→29.54) nhưng **std lại tăng** (23.33→27.87).

## 4. Tinh chỉnh kiến trúc: sâu hơn vs rộng hơn

- **Deeper** (13→13→**6**→1, thêm 1 hidden layer): **-22.83 (25.33) MSE**.
- **Wider** (13→**20**→1, tăng gấp ~1.5 lần neuron ở layer duy nhất, giữ nguyên số layer): **-21.71 (24.39) MSE** — cải thiện **hơn cả deeper**, cho kết quả tốt nhất trong 4 thí nghiệm.

## 5. Tổng hợp 4 thí nghiệm

| Thí nghiệm | Kiến trúc | MSE (mean) | Std |
|---|---|---|---|
| Baseline | 13→13→1 | 32.65 | 23.33 |
| Standardized | 13→13→1 (đã chuẩn hoá) | 29.54 | 27.87 |
| Deeper | 13→13→6→1 (đã chuẩn hoá) | 22.83 | 25.33 |
| Wider | 13→20→1 (đã chuẩn hoá) | **21.71** | 24.39 |


## 6. Vận dụng


**11.1–11.6** — Model baseline (13→13→1) + đánh giá bằng KerasRegressor + k-fold

In [1]:
# Regression Example With Boston Dataset: Baseline
from pandas import read_csv
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
# load dataset
dataframe = read_csv("housing.csv", delim_whitespace=True, header=None)
dataset = dataframe.values
# split into input (X) and output (Y) variables
X = dataset[:,0:13]
Y = dataset[:,13]
# define base model
def baseline_model():
    # create model
    model = Sequential()
    model.add(Dense(13, input_shape=(13,),
                    kernel_initializer='normal', activation='relu'))
    model.add(Dense(1, kernel_initializer='normal'))
    # Compile model
    model.compile(loss='mean_squared_error', optimizer='adam')
    return model
# evaluate model
estimator = KerasRegressor(model=baseline_model, epochs=100, batch_size=5, verbose=0)
kfold = KFold(n_splits=10)
results = cross_val_score(estimator, X, Y, cv=kfold, scoring='neg_mean_squared_error')
print("Baseline: %.2f (%.2f) MSE" % (results.mean(), results.std()))


2026-09-06 07:10:18.079423: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-06 07:10:18.195493: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-06 07:10:18.195543: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-06 07:10:18.200602: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-06 07:10:18.221012: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-06 07:10:18.222932: I tensorflow/core/platform/cpu_feature_guard.cc:1

Baseline: -46.17 (32.36) MSE


**11.7–11.8** — Thêm StandardScaler vào Pipeline

In [2]:
# Regression Example With Boston Dataset: Standardized
from pandas import read_csv
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
# load dataset
dataframe = read_csv("housing.csv", delim_whitespace=True, header=None)
dataset = dataframe.values
# split into input (X) and output (Y) variables
X = dataset[:,0:13]
Y = dataset[:,13]
# define base model
def baseline_model():
    # create model
    model = Sequential()
    model.add(Dense(13, input_shape=(13,),
                    kernel_initializer='normal', activation='relu'))
    model.add(Dense(1, kernel_initializer='normal'))
    # Compile model
    model.compile(loss='mean_squared_error', optimizer='adam')
    return model
# evaluate model with standardized dataset
estimators = []
estimators.append(('standardize', StandardScaler()))
estimators.append(('mlp', KerasRegressor(model=baseline_model,
                                         epochs=50, batch_size=5, verbose=0)))
pipeline = Pipeline(estimators)
kfold = KFold(n_splits=10)
results = cross_val_score(pipeline, X, Y, cv=kfold, scoring='neg_mean_squared_error')
print("Standardized: %.2f (%.2f) MSE" % (results.mean(), results.std()))


/tmp/ipykernel_21818/266250902.py:11: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  dataframe = read_csv("housing.csv", delim_whitespace=True, header=None)


Standardized: -30.03 (27.53) MSE


**11.9–11.11** — Kiến trúc sâu hơn (13→13→6→1)

In [3]:
# Regression Example With Boston Dataset: Standardized and Larger
from pandas import read_csv
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
# load dataset
dataframe = read_csv("housing.csv", delim_whitespace=True, header=None)
dataset = dataframe.values
# split into input (X) and output (Y) variables
X = dataset[:,0:13]
Y = dataset[:,13]
# define the model
def larger_model():
    # create model
    model = Sequential()
    model.add(Dense(13, input_shape=(13,),
                    kernel_initializer='normal', activation='relu'))
    model.add(Dense(6, kernel_initializer='normal', activation='relu'))
    model.add(Dense(1, kernel_initializer='normal'))
    # Compile model
    model.compile(loss='mean_squared_error', optimizer='adam')
    return model
# evaluate model with standardized dataset
estimators = []
estimators.append(('standardize', StandardScaler()))
estimators.append(('mlp', KerasRegressor(model=larger_model,
                                         epochs=50, batch_size=5, verbose=0)))
pipeline = Pipeline(estimators)
kfold = KFold(n_splits=10)
results = cross_val_score(pipeline, X, Y, cv=kfold, scoring='neg_mean_squared_error')
print("Larger: %.2f (%.2f) MSE" % (results.mean(), results.std()))


/tmp/ipykernel_21818/3294065147.py:11: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  dataframe = read_csv("housing.csv", delim_whitespace=True, header=None)


Larger: -21.66 (22.42) MSE


**11.12–11.14** — Kiến trúc rộng hơn (13→20→1)

In [4]:
# Regression Example With Boston Dataset: Standardized and Wider
from pandas import read_csv
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
# load dataset
dataframe = read_csv("housing.csv", delim_whitespace=True, header=None)
dataset = dataframe.values
# split into input (X) and output (Y) variables
X = dataset[:,0:13]
Y = dataset[:,13]
# define wider model
def wider_model():
    # create model
    model = Sequential()
    model.add(Dense(20, input_shape=(13,),
                    kernel_initializer='normal', activation='relu'))
    model.add(Dense(1, kernel_initializer='normal'))
    # Compile model
    model.compile(loss='mean_squared_error', optimizer='adam')
    return model
# evaluate model with standardized dataset
estimators = []
estimators.append(('standardize', StandardScaler()))
estimators.append(('mlp', KerasRegressor(model=wider_model,
                                         epochs=100, batch_size=5, verbose=0)))
pipeline = Pipeline(estimators)
kfold = KFold(n_splits=10)
results = cross_val_score(pipeline, X, Y, cv=kfold, scoring='neg_mean_squared_error')
print("Wider: %.2f (%.2f) MSE" % (results.mean(), results.std()))


/tmp/ipykernel_21818/3956193735.py:11: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  dataframe = read_csv("housing.csv", delim_whitespace=True, header=None)


Wider: -20.85 (23.11) MSE
